# v10-build-data — dựng lại sft.jsonl + dev.jsonl: prompt HYBRID (BM25 + e5 + RRF)

**Phiên 5.** Infer đã chuyển sang hybrid (`infer_hybrid.ipynb`), còn sft.jsonl phiên 3
dựng bằng BM25 thuần. Train và thi phải thấy cùng một bộ lấy Evidence, nên dựng lại
cả hai tập với `--dense` — cùng `nap_dense` mà infer.py import, không phải bản sao.

---

Lịch sử phiên 3 (vẫn đúng):

Hai thay đổi trong một lần chạy (setup + nạp corpus ~10 phút trả một lần thay vì hai).

**1. Số hiệu văn bản vào prompt.** Đo trên 120 câu dev của eval b318 (METEOR 0,5982):
78/120 câu model bịa ra năm ban hành **không hề có trong prompt**. `Evidence.doc_title`
là dòng ALL-CAPS crawl HTML, không mang số hiệu lẫn năm — trong khi `Document.number`
khớp nguyên văn đáp án 65% số câu. Commit `b8b97d2` nối đường đó.

**2. Bỏ `max_chars`, mỗi câu lấy đủ `top_n=5` Evidence.** Quyết định của người dùng
2026-09-14. Trần 12.000 ký tự cắn TRƯỚC `top_n` (3,36 Evidence/câu chứ không phải 5),
tức nó mới là thứ đang vứt bằng chứng — rồi mẫu bị vứt bằng chứng lại bị đếm vào
21,3% `retrieval_truot_gold`.

Đo lại 300 câu (BM25, tokenizer Qwen2, `scratchpad/do_khong_gioi_han.py`):

| | ev/câu | ký tự p50 | seq p50 | seq p99 | seq max | >4.096 | >6.144 | >8.192 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 12.000 (cũ) | 3,36 | 10.802 | 3.442 | 4.847 | 6.162 | 9,7% | 0,3% | 0,0% |
| **không trần** | **4,99** | **16.403** | **5.071** | **8.244** | **8.377** | **84,3%** | **18,3%** | **1,0%** |

Có trần trên: `top_n=5` chặn ở ~24.000 ký tự, không phình vô hạn. Vì thế
`train_sft.py --max-seq` lên **10.240** (6.144 sẽ âm thầm vứt 18,3% mẫu). Cái giá chưa
đo được: activation bs=1 tăng ~1,7x trên T4 16 GB.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Chỉ nhận codebase từ dataset `legalqa-v10-src`. `kernel_sources` mang theo cả `repo/`
# của lần chạy trước và glob bắt trúng bản CŨ đó trước — v10-train-sft v2 chết vì
# `scripts/train_sft.py` không tồn tại trong bản nằm trong output kernel cũ.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
if SRC is None:
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src")
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())

# Cả kernel này chỉ tồn tại để prompt có số hiệu. Chạy 3,5 giờ GPU rồi mới phát hiện
# dataset đẩy nhầm bản cũ là mất nguyên một ngày quota — chặn ngay ở đây.
from legalqa.evidence import Evidence
assert "doc_number" in Evidence.__slots__, "codebase CŨ: chưa có Evidence.doc_number"
print("Evidence.doc_number: OK")

# Nghiệm thu thứ hai: trần ký tự đã bỏ. Chạy 3,5 giờ GPU ra đúng file cũ vì dataset
# chưa propagate là mất một ngày quota — cùng lý do với assert ở trên.
sys.path.insert(0, "scripts")
from build_sft import MAX_CHARS, TOP_N
assert TOP_N == 5 and MAX_CHARS > 10**6, f"codebase CŨ: top_n={TOP_N} max_chars={MAX_CHARS}"
print(f"top_n={TOP_N} max_chars={MAX_CHARS:,} (không trần): OK")


In [ ]:
!pip -q install transformers==4.44.2 sentencepiece


In [ ]:
cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

# clean/ (chunks + bm25 + corpus + gold) vào bằng dataset `legalqa-clean`, không dựng lại:
# dựng mất 26 phút, và kernel bật GPU thì 26 phút đó đốt quota GPU dù không chạm GPU.
# Output kernel KHÔNG chia sẻ được giữa hai tài khoản nên đường cũ (kernel_sources) chết
# sau khi đổi tài khoản — dataset thì chia sẻ được.
ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — chưa gắn dataset legalqa-clean"
CU = ch.parent

# Symlink sang working: build_sft.py ghi bm25.pkl/sft.jsonl vào CLEAN nếu ghi được, mà
# /kaggle/input là read-only.
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in CU.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CLEAN_DIR"] = str(CLEAN)
os.environ["PYTHONPATH"] = "src"

print("DATA_DIR", DATA_DIR, "|", len(list((DATA_DIR / "selected-contexts").glob("*.json"))), "file corpus")
for f in sorted(CLEAN.iterdir()):
    print(f"  {f.name:22} {f.stat().st_size/1e6:9.1f} MB")
for n in ("chunks.jsonl", "bm25.pkl", "corpus.jsonl", "gold.jsonl",
          "train_sft.jsonl", "warmup_sft.jsonl", "gold_warmup.jsonl"):
    assert (CLEAN / n).exists(), f"thiếu {n} trong legalqa-clean"


In [ ]:
import json

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    MODEL = str(local[0])
    print("dùng bản đã gắn:", MODEL)
else:
    from huggingface_hub import snapshot_download
    MODEL = snapshot_download("AITeamVN/Vietnamese_Reranker")
    print("tải từ HuggingFace:", MODEL)

tk = next((p for p in Path("/kaggle/input").glob("**/tokenizer.json") if "qwen" in str(p).lower()), None)
assert tk, "Không thấy tokenizer Qwen — gắn model qwen3-5-2b-base"
os.environ["QWEN_TOKENIZER"] = str(tk)
print("tokenizer:", tk)

# Bi-encoder cho nhánh dense. PHẢI trùng model `infer_hybrid.ipynb` dùng: lượt 3/4 train
# trên prompt BM25 thuần rồi đem đi infer với hybrid là lệch phân phối input đúng kiểu đã
# làm phiên 2 bịa năm ban hành — model học chép từ một tập Evidence khác tập nó gặp khi thi.
e5 = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
      if "multilingual-e5-small" in str(p)]
if e5:
    DENSE = str(e5[0])
else:
    from huggingface_hub import snapshot_download
    DENSE = snapshot_download("intfloat/multilingual-e5-small")
print("dense:", DENSE)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")


In [ ]:
# subprocess.run(check=True), KHÔNG dùng `!python`: trong IPython `!python` không ném
# theo mã thoát, script chết giữa chừng mà cell vẫn xanh và cell sau đọc file rỗng.
subprocess.run([sys.executable, "scripts/build_sft.py",
    "--model", MODEL, "--dense", DENSE, "--batch-size", "32", "--pool", "50",
    "--ra", "/kaggle/working/sft.jsonl"], check=True)


In [ ]:
subprocess.run([sys.executable, "scripts/build_sft.py",
    "--model", MODEL, "--dense", DENSE, "--batch-size", "32", "--pool", "50",
    "--nguon", "warmup_sft.jsonl", "--gold", "gold_warmup.jsonl", "--giu-het",
    "--ra", "/kaggle/working/dev.jsonl"], check=True)


In [ ]:
import re
from collections import Counter

# Nghiệm thu ĐÚNG thứ đã khẳng định: không phải "file có tồn tại" mà "prompt có số hiệu".
SO_HIEU = re.compile(r"\d+[/-]\d+")
for ten in ("sft.jsonl", "dev.jsonl"):
    rows = [json.loads(l) for l in open(f"/kaggle/working/{ten}", encoding="utf-8")]
    co = sum(1 for r in rows if SO_HIEU.search(r["prompt"].split("Câu hỏi:")[0]))
    sl = sorted(r["seq_len"] for r in rows)
    ck = sorted(len(r["prompt"]) for r in rows)
    print(f"{ten}: {len(rows):,} mẫu | prompt có số hiệu {co}/{len(rows)} = {co/len(rows):.0%}")
    print(f"  seq_len  p50 {sl[len(sl)//2]}  p95 {sl[int(.95*len(sl))]}  max {sl[-1]}  >10240 {sum(1 for x in sl if x>10240)}  (cổng train --max-seq 10.240)")
    print(f"  ký tự    p50 {ck[len(ck)//2]}  max {ck[-1]}  (KHÔNG còn trần ký tự; top_n=5 là ràng buộc duy nhất)")
    for k, v in Counter(r["ly_do_loai"] or "GIỮ" for r in rows).most_common():
        print(f"    {k:22} {v:5}  {100*v/len(rows):5.1f}%")

print()
print(json.loads(open("/kaggle/working/dev.jsonl", encoding="utf-8").readline())["prompt"][:400])
